# DINOv3 — Chinee apple weed detection (Colab)

Self-contained Colab notebook for the `dinov3-colab` experiment in `weed-detection-experiments`.

What it does:
1. Clones the official [facebookresearch/dinov3](https://github.com/facebookresearch/dinov3) repo.
2. Loads DINOv3 weights from your Google Drive.
3. **Unsupervised (§4–5):** extracts per-patch features and finds the foreground via [CLS]-token saliency, with a Gradio UI to box the dominant object. Backbone-only — no class label.
4. **Supervised linear probe (§6–7):** trains a logistic-regression head on the *frozen* backbone from your labeled crops to classify Chinee apple vs other trees, saves it to Drive, and applies it per-patch for class-aware detection heatmaps.

**Before running:** Runtime → Change runtime type → **GPU** (T4 free is fine).

**One-time:** accept the DINOv3 license at https://ai.meta.com/resources/models-and-libraries/dinov3-downloads/ and place the `.pth` in your Drive (default expected path is shown in step 2).

**Note:** the `.pth` is the finished self-supervised backbone — it is *never* retrained. §6 only fits a small head on labeled crops; no pre-training images are needed from you.

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Mount Google Drive

Weights are read from Drive so you don't re-upload them each session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

Adjust paths if your Drive layout differs.

In [ ]:
import os

DINOV3_REPO = '/content/dinov3'
WEIGHTS_PATH = '/content/drive/MyDrive/dinov3/weights/dinov3_vitb16_pretrain_lvd1689m.pth'
ARCH = 'dinov3_vitb16'

assert os.path.isfile(WEIGHTS_PATH), f'Weights not found at {WEIGHTS_PATH} — update WEIGHTS_PATH or copy the .pth into Drive.'
print('Weights OK:', WEIGHTS_PATH)

## 3. Clone DINOv3 and install dependencies

In [ ]:
if not os.path.isdir(DINOV3_REPO):
    !git clone --depth 1 https://github.com/facebookresearch/dinov3.git {DINOV3_REPO}

%pip install -q pillow scipy scikit-learn torchmetrics 'gradio>=4.0'

## 4. Inference code

Same logic as `dinov3_detect.py`, inlined so the notebook is self-contained.

In [ ]:
import numpy as np
import torch
from PIL import Image, ImageDraw
from scipy.ndimage import find_objects, label
from torchvision import transforms

PATCH = 16
IMG_SIZE = 768
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

model = torch.hub.load(DINOV3_REPO, ARCH, source='local', weights=WEIGHTS_PATH).to(DEVICE).eval()

_tx = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

@torch.inference_mode()
def cls_saliency(image):
    # Cosine similarity between [CLS] and each patch token.
    # Higher = more aligned with the image's global object representation.
    x = _tx(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    out = model.forward_features(x)
    cls = out['x_norm_clstoken'][0]
    patches = out['x_norm_patchtokens'][0]
    sim = torch.nn.functional.cosine_similarity(patches, cls.unsqueeze(0), dim=-1)
    grid = IMG_SIZE // PATCH
    return sim.float().cpu().numpy().reshape(grid, grid)

def foreground_mask(sim):
    s = (sim - sim.min()) / (sim.max() - sim.min() + 1e-8)
    return (s > s.mean()).astype(np.uint8)

def largest_component_bbox(mask):
    lbl, n = label(mask)
    if n == 0:
        return None
    sizes = np.bincount(lbl.ravel())
    sizes[0] = 0
    idx = int(sizes.argmax())
    sl = find_objects(lbl == idx)[0]
    return sl[1].start, sl[0].start, sl[1].stop, sl[0].stop

def annotate(image, bbox_grid, grid_size):
    if bbox_grid is None:
        return image
    W, H = image.size
    sx, sy = W / grid_size, H / grid_size
    x0, y0, x1, y1 = bbox_grid
    out = image.copy()
    ImageDraw.Draw(out).rectangle([x0 * sx, y0 * sy, x1 * sx, y1 * sy], outline='red', width=4)
    return out

def infer(image):
    sim = cls_saliency(image)
    bbox = largest_component_bbox(foreground_mask(sim))
    return annotate(image, bbox, sim.shape[0])

## 5. Launch the Gradio UI

Click the public `*.gradio.live` URL to open the interface in a new tab.

In [ ]:
import gradio as gr

gr.Interface(
    fn=infer,
    inputs=gr.Image(type='pil', label='Upload'),
    outputs=gr.Image(type='pil', label='Detected object'),
    title='DINOv3 object localization — Chinee apple',
    description='Foreground is found via PCA over DINOv3 patch tokens; the largest connected blob is boxed. Backbone-only — no class label.',
).launch(share=True)

## 6. Supervised linear probe — classify Chinee apple vs other trees

The cells above are **backbone-only** (no class label — they just find "something"). This section
trains a tiny classifier **on top of the frozen DINOv3 features** so the model can actually name
Chinee apple and tell it apart from other trees.

**Nothing in DINOv3 is retrained.** We only fit a logistic-regression head on extracted features —
seconds on a GPU. You provide a small set of **labeled crops** (~30–50 per class to prototype,
~100–200 for a usable classifier), sorted into one folder per class. No pre-training, no boxes.

In [ ]:
# Drive folder with one subfolder per class, each holding cropped images:
#   dataset/
#   ├── chinee_apple/   crops centered on Chinee apple canopy
#   ├── other_tree/     crops of other trees/shrubs to distinguish from
#   └── background/     grass, soil, sky (optional, improves rejection)
DATASET_DIR = '/content/drive/MyDrive/dinov3/dataset'

CLASSES = sorted(
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
)
print('Classes:', CLASSES)
for c in CLASSES:
    n = len([f for f in os.listdir(os.path.join(DATASET_DIR, c))
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'  {c}: {n} images')


In [ ]:
import numpy as np
from PIL import Image

@torch.inference_mode()
def patch_tokens(image):
    # (num_patches, dim) L2-normalized patch tokens for one image.
    x = _tx(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    out = model.forward_features(x)
    p = out['x_norm_patchtokens'][0]
    return torch.nn.functional.normalize(p, dim=-1).float().cpu().numpy()

def image_embedding(image):
    # Mean-pooled patch tokens — SAME feature space used per-patch for the heatmap,
    # so the trained classifier transfers directly to dense (per-patch) inference.
    return patch_tokens(image).mean(0)

# Extract one embedding per labeled image.
X, y = [], []
for label_idx, c in enumerate(CLASSES):
    folder = os.path.join(DATASET_DIR, c)
    for f in sorted(os.listdir(folder)):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        X.append(image_embedding(Image.open(os.path.join(folder, f))))
        y.append(label_idx)
X = np.stack(X); y = np.array(y)
print('Features:', X.shape, ' Labels:', y.shape, ' per class:', np.bincount(y))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

# Frozen-feature linear probe. class_weight balances uneven folder sizes.
clf = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced')
clf.fit(X_tr, y_tr)

y_pred = clf.predict(X_te)
print(classification_report(y_te, y_pred, target_names=CLASSES))

cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha='center', va='center')
plt.title('Confusion matrix'); plt.tight_layout(); plt.show()


### 6b. Save / reload the trained probe

Persists just the logistic-regression head + class names (~KB) to Drive. Reload it next session to
skip retraining — re-running the frozen backbone (§1–4) is all you need alongside it.

In [ ]:
import joblib

# Persist the probe to Drive so you don't retrain each session.
# The backbone weights are NOT saved here — only the small head + class names.
PROBE_PATH = '/content/drive/MyDrive/dinov3/chinee_probe.joblib'

joblib.dump({'clf': clf, 'classes': CLASSES, 'img_size': IMG_SIZE, 'patch': PATCH}, PROBE_PATH)
print('Saved probe ->', PROBE_PATH)

# --- To reload in a later session, skip the §6 feature-extraction and training
#     cells and run this instead: ---
# bundle = joblib.load(PROBE_PATH)
# clf, CLASSES = bundle['clf'], bundle['classes']
# print('Loaded probe for classes:', CLASSES)


## 7. Class-aware detection — Chinee apple heatmap

The probe was trained on **patch-token features**, so we can run it on **every patch** of a new
image to get a per-class probability map. High-probability Chinee apple patches are boxed.

This turns the image-level classifier into a detector **with no extra training** — the same head,
applied densely. Upload an image to see: input | P(chinee apple) overlay | box.

In [ ]:
import io
from google.colab import files

# Index of the Chinee apple class in CLASSES (falls back to 0 if named differently).
CHINEE_IDX = CLASSES.index('chinee_apple') if 'chinee_apple' in CLASSES else 0

@torch.inference_mode()
def chinee_heatmap(image):
    # Apply the trained probe to EVERY patch token -> P(chinee apple) per patch.
    # Works because the probe was trained on mean-pooled patch tokens (same space).
    p = patch_tokens(image)
    prob = clf.predict_proba(p)[:, CHINEE_IDX]
    grid = IMG_SIZE // PATCH
    return prob.reshape(grid, grid)

uploaded = files.upload()
filename = next(iter(uploaded))
img = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')

heat = chinee_heatmap(img)
pred = CLASSES[int(clf.predict(image_embedding(img)[None])[0])]
mask = (heat > 0.5).astype(np.uint8)
bbox_grid = largest_component_bbox(mask)

heat_big = Image.fromarray((heat * 255).astype(np.uint8)).resize(img.size, Image.BILINEAR)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img); axes[0].set_title(f'Input — predicted: {pred}'); axes[0].axis('off')
axes[1].imshow(img); axes[1].imshow(heat_big, cmap='inferno', alpha=0.5)
axes[1].set_title('P(chinee apple) per patch'); axes[1].axis('off')
axes[2].imshow(img)
if bbox_grid is not None:
    W, H = img.size
    sx, sy = W / heat.shape[0], H / heat.shape[0]
    x0, y0, x1, y1 = bbox_grid
    axes[2].add_patch(plt.Rectangle((x0 * sx, y0 * sy), (x1 - x0) * sx, (y1 - y0) * sy, fill=False, edgecolor='lime', linewidth=3))
axes[2].set_title('Chinee apple box (p>0.5)'); axes[2].axis('off')
plt.tight_layout(); plt.show()


## 8. CAFe-DINO — open-vocabulary weed detection (no training)

Sections 6–7 trained a supervised probe. This runs **CAFe-DINO**
([DINO Soars](https://github.com/rfaulk/DINO_Soars), CVPRW 2026) instead: prompt
the pretrained model with `"chinee apple"` + distractor classes and get a
full-resolution weed mask — **no labels, no training**.

Same self-contained style as the rest of this notebook. Uses the **ViT-L/16**
DINOv3 backbone (not the ViT-B/16 from Section 2).

### 8a. Configuration — ViT-L/16 weights

In [ ]:
import os

DINOSOARS_REPO = '/content/DINO_Soars'

# CAFe-DINO needs the gated ViT-L/16 DINOv3 weights (accept Meta's license and
# download BOTH files). The ViT-B weights from Section 2 will NOT work here:
# this is ViT-L/16 (~1.1 GB backbone), not the 342 MB dinov3_vitb16 file.
WEIGHTS_DIR = '/content/drive/MyDrive/dinov3/weights'
os.environ['DINOV3_VITL16_WEIGHTS']  = f'{WEIGHTS_DIR}/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'
os.environ['DINOV3_DINOTXT_WEIGHTS'] = f'{WEIGHTS_DIR}/dinov3_vitl16_dinotxt_vision_head_and_text_encoder-a442d8f5.pth'

# 1. Drive mounted? (a "missing" file is often just an unmounted Drive)
if not os.path.isdir('/content/drive/MyDrive'):
    raise RuntimeError('Google Drive is not mounted. Run the "1. Mount Google Drive" cell first, then re-run this cell.')

# 2. Report precisely which file(s) are missing instead of a bare AssertionError.
missing = [v for v in ('DINOV3_VITL16_WEIGHTS', 'DINOV3_DINOTXT_WEIGHTS')
           if not os.path.isfile(os.environ[v])]
if missing:
    print(f'Contents of {WEIGHTS_DIR}:')
    found = sorted(os.listdir(WEIGHTS_DIR)) if os.path.isdir(WEIGHTS_DIR) else []
    print('  (empty / dir not found)' if not found else
          '\n'.join(f'  {f}  ({os.path.getsize(os.path.join(WEIGHTS_DIR, f))/1e9:.2f} GB)' for f in found))
    names = '\n'.join(f'  - {v}: {os.path.basename(os.environ[v])}' for v in missing)
    raise FileNotFoundError(
        'Missing gated DINOv3 ViT-L/16 weight file(s):\n' + names +
        '\n\nDownload the exact file(s) above (accept the license, then grab the\n'
        'ViT-L/16 LVD-1689M files) from\n'
        '  https://ai.meta.com/resources/models-and-libraries/dinov3-downloads/\n'
        'and copy them into the folder above with those exact names, e.g.:\n'
        "  URL = 'PASTE_SIGNED_URL'\n"
        f"  !wget -c -O '{os.environ['DINOV3_VITL16_WEIGHTS']}' \"$URL\"\n"
        'Note: the 342 MB dinov3_vitb16 file from Section 2 is a different model '
        'and cannot be used here.'
    )

print('DINOv3 ViT-L/16 weights OK')
print(' ', os.environ['DINOV3_VITL16_WEIGHTS'])
print(' ', os.environ['DINOV3_DINOTXT_WEIGHTS'])

### 8b. Clone DINO_Soars, patch it, install deps, fetch trained weights

The upstream repo hardcodes two developer weight paths (`/home/rfaulken/...`)
in `dinov3/hub/backbones.py` and `dinov3/hub/dinotxt.py`; we rewrite them to read
the env vars set above.

In [ ]:
import re

if not os.path.isdir(DINOSOARS_REPO):
    !git clone --depth 1 https://github.com/rfaulk/DINO_Soars.git {DINOSOARS_REPO}

# --- patch the two hardcoded torch.load paths ---
for rel, var, tag in [('dinov3/hub/backbones.py',  'DINOV3_VITL16_WEIGHTS', 'lvd1689m'),
                      ('dinov3/hub/dinotxt.py',    'DINOV3_DINOTXT_WEIGHTS', 'dinotxt')]:
    p = os.path.join(DINOSOARS_REPO, rel)
    s = open(p).read()
    s = re.sub(r"torch\.load\('/home/rfaulken/[^']*" + tag + r"[^']*'\)",
               f"torch.load(os.environ['{var}'], map_location='cpu')", s)
    if not re.search(r'^import os$', s, re.M):
        s = 'import os\n' + s
    open(p, 'w').write(s)
print('patched hardcoded paths')

%pip install -q einops timm 'albumentations>=2.0' omegaconf ftfy regex opencv-python-headless tifffile huggingface_hub

# trained CAFe-DINO checkpoint (no RS training needed)
from huggingface_hub import snapshot_download
CKPT_DIR = snapshot_download(repo_id='rfaulken/cafedino',
                             local_dir=os.path.join(DINOSOARS_REPO, 'checkpoints'))
import glob
CAFEDINO_WEIGHTS = sorted(glob.glob(f'{CKPT_DIR}/*.pth'))[0]
print('CAFe-DINO checkpoint:', CAFEDINO_WEIGHTS)

### 8c. Load the model

In [ ]:
import sys, torch, torch.nn.functional as F
for p in (DINOSOARS_REPO, f'{DINOSOARS_REPO}/CAFe_DINO', f'{DINOSOARS_REPO}/anyup'):
    if p not in sys.path:
        sys.path.insert(0, p)

from dinov3.hub.dinotxt import dinov3_vitl16_dinotxt_tet1280d20h24l
from CAFe_DINO.modeling.cafedino import CAFe_DINO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_float32_matmul_precision('high')

backbone, tokenizer = dinov3_vitl16_dinotxt_tet1280d20h24l()
backbone.to(DEVICE).eval()
upsampler = torch.hub.load(f'{DINOSOARS_REPO}/anyup', 'anyup', source='local', verbose=False).to(DEVICE).eval()

ckpt = torch.load(CAFEDINO_WEIGHTS, map_location='cpu')
sd = ckpt.get('model', ckpt)
sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
AGG_DIM = int(sd['corr_embed.weight'].shape[0])   # infer aggregator width from ckpt

cafedino = CAFe_DINO(backbone, tokenizer, upsampler, input_resolution=(14, 14),
                     device=DEVICE, aggregator_dim=AGG_DIM).to(DEVICE)
cafedino.load_state_dict(sd, strict=False)
cafedino.eval()
print('CAFe-DINO loaded (aggregator_dim=%d) on %s' % (AGG_DIM, DEVICE))

### 8d. Detect chinee apple on an uploaded tile

`chinee apple` is the target (index 0); the rest are distractors so the softmax
has competing land cover. Prompt ensembling + strided 224/112 windows follow the
paper. Tune the threshold and prompt wording on your imagery.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)
TEMPLATES = ('a photo of {}', 'an image of {}', 'a photo of a {}', 'an aerial image of {}',
             'a drone image of {}', 'a satellite image of {}', 'a close-up photo of {}')

@torch.no_grad()
def text_embed(class_names):
    prompts = [t.format(n) for n in class_names for t in TEMPLATES]
    toks = tokenizer.tokenize(prompts).to(DEVICE)
    e = backbone.encode_text(toks)[:, 1024:]
    e = e.view(len(class_names), len(TEMPLATES), -1).mean(1)
    return F.normalize(e, p=2, dim=1)

@torch.no_grad()
def strided(img, temb, C, side=224, stride=112):
    _, _, H, W = img.shape
    probs = torch.zeros(C, H, W, device=DEVICE); counts = torch.zeros(H, W, device=DEVICE)
    hg = max(H - side + stride - 1, 0) // stride + 1
    wg = max(W - side + stride - 1, 0) // stride + 1
    for i in range(hg):
        for j in range(wg):
            y1, x1 = i * stride, j * stride
            y2, x2 = min(y1 + side, H), min(x1 + side, W); y1, x1 = max(y2 - side, 0), max(x2 - side, 0)
            out = cafedino(img[:, :, y1:y2, x1:x2], temb, pre_text_emb=True).squeeze(0)
            probs[:, y1:y2, x1:x2] += out; counts[y1:y2, x1:x2] += 1
    return probs / counts.clamp(min=1)

@torch.no_grad()
def detect(path, class_names, resize=512, thresh=0.5):
    pil = Image.open(path).convert('RGB').resize((resize, resize), Image.BILINEAR)
    arr = (np.asarray(pil, np.float32) / 255.0 - MEAN) / STD
    x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    temb = text_embed(class_names)
    with torch.amp.autocast('cuda') if DEVICE == 'cuda' else torch.no_grad():
        logits = strided(x, temb, len(class_names))
    prob = torch.softmax(logits.float(), 0)
    return pil, prob[0].cpu().numpy(), (prob[0].cpu().numpy() >= thresh)

from google.colab import files
up = files.upload(); fname = next(iter(up)); open('/content/_tile.jpg', 'wb').write(up[fname])

CLASSES = ['chinee apple', 'grass', 'tree', 'bare soil', 'shrub', 'dry grass', 'road']
base, heat, mask = detect('/content/_tile.jpg', CLASSES, resize=512, thresh=0.5)

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(base); ax[0].set_title('Input UAV tile'); ax[0].axis('off')
ax[1].imshow(base); ax[1].imshow(heat, cmap='inferno', alpha=0.55); ax[1].set_title('P(chinee apple)'); ax[1].axis('off')
ax[2].imshow(base); ax[2].imshow(np.ma.masked_where(~mask, mask), cmap='autumn', alpha=0.6)
ax[2].set_title(f'weed mask (>{0.5:.2f}) — {100*mask.mean():.1f}% of tile'); ax[2].axis('off')
plt.tight_layout(); plt.show()